# GPA778 - Laboratory 2
# Part B - Perception II: 3D Representation by 2D Lidar Scans

During the last laboratory (2a), you created the `jerro_perception` package, with a launch file and two nodes (`cam_traitement` and `lidar_traitement`). These files allowed the acquisition of perception data to begin, but little work has been done yet regarding the processing of this data.

In addition, for the moment, the robot's onboard computer (the RPI) has only acquired 2D perception data. Although it is possible to perform autonomous navigation from 2D perception data, we now want to obtain a more complete perception of the environment thanks to 3D perception.

**Objective:** Reconstruct the environment near the robot in 3D, thanks to the acquisition of a succession of 2D lidar scans.

## 2.2.1 Academic Objectives

The objective of this laboratory is to succeed in acquiring a succession of 2D lidar scans in order to create a 3D representation of the environment near the robot. To do this, it will of course be necessary to rotate the lidar's acquisition plane, using the **servomotor** allowing this task to be accomplished.

### Specific objectives:

- Develop the projection model from pseudo-spherical data to Cartesian data, expressed in the robot's frame (`base_link`), and **verify** it with the TF chain published from the URDF
- Become familiar with the concept of an **"action"** under ROS, by adding an action to the `lidar_traitement.py` node and coding an associated callback
- Control the servomotor via a ROS **service** (`/set_servo_pos`)
- Use the **`ScanAssembler`** class (`laser_assembler` package) to assemble a succession of `LaserScan` messages into a `PointCloud2`, using TF transformations
- Deepen the concept of TF and the use of the [joint_state_publisher](https://github.com/ros/joint_state_publisher/blob/ros2/joint_state_publisher/README.md) package

## 2.2.2 Milestone 1: Development of the projection model from pseudo-spherical to Cartesian data in the robot's frame

### Problem statement

Since the axis of rotation rotating the lidar does not pass through its optical center, we end up with **pseudo-spherical coordinates**, thus preventing us from using well-known projection formulas.

It is therefore necessary to develop the equations allowing lidar points to be expressed in the **robot's frame (`base_link`)**.

The robot's reference frames are described in the URDF file of the `jerro_description` package (`urdf/jerro.urdf`). As seen in laboratory 1c, `robot_state_publisher` calculates the transformations between reference frames from this URDF and publishes them in TF. In this milestone, you develop the model by hand; you will then compare it to what TF publishes.

### Identification of the transformation matrix R_T_L

We seek to identify the homogeneous transformation matrix $^RT_L$, where:
- **R** refers to the robot frame
- **L** refers to the lidar frame (origin at the optical center)

![Figure 2.4 - Lidar Frame](figure_2_4.png)
*Figure 2.4: Lidar frame, as specified by the manufacturer. Left: top view and right: front view.*

![Figure 2.5 - Frame Definition](figure_2_5.png)
*Figure 2.5: Definition of relevant frames, which will allow identifying R_T_L*

### Defined frames

On the image, several frames are defined:
- **Frame L**: Lidar frame (as indicated by the manufacturer)
- **Frame R**: Robot frame
- **Frame S**: Additional frame with origin at the intersection of the servomotor's rotation axis and the $x_R - z_R$ plane

![Figure 2.6 - Geometric Relationships](figure_2_6.png)
*Figure 2.6: Identification of relevant frames as well as their geometric relationships.*

**Important note:** The lidar is rotated by an angle $\psi$ relative to the servomotor. When $\psi = 0$, the $z_L$ axis is collinear with the $x_S$ axis and points in the opposite direction of the latter.

In practice, we will not allow the lidar to rotate at such small angles, because it would then collide with the RPI housing.

### Homogeneous transformation equations

To go from frame R to frame L, homogeneous transformations are used:

$$^RT_L = Trans_{xyz}(0.04, 0, 0.034) \cdot Rot_y(\psi - \pi/2) \cdot Trans_{xyz}(0, 0, 0.113)$$

More explicitly:

$$^RT_L = \begin{bmatrix}
1 & 0 & 0 & 0.04 \\
0 & 1 & 0 & 0 \\
0 & 0 & 1 & 0.034 \\
0 & 0 & 0 & 1
\end{bmatrix}
\begin{bmatrix}
\cos(\psi - \pi/2) & 0 & \sin(\psi - \pi/2) & 0 \\
0 & 1 & 0 & 0 \\
-\sin(\psi - \pi/2) & 0 & \cos(\psi - \pi/2) & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
\begin{bmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 1 & 0.113 \\
0 & 0 & 0 & 1
\end{bmatrix}$$

### Final result

By performing matrix multiplications and simplifying, we get:

$$^RT_L = \begin{bmatrix}
\cos(\psi - \pi/2) & 0 & \sin(\psi - \pi/2) & 0.04 + 0.113 \cdot \sin(\psi - \pi/2) \\
0 & 1 & 0 & 0 \\
-\sin(\psi - \pi/2) & 0 & \cos(\psi - \pi/2) & 0.034 + 0.113 \cdot \cos(\psi - \pi/2) \\
0 & 0 & 0 & 1
\end{bmatrix}$$

Compact form:

$$^RT_L = \begin{bmatrix}
Rot(y, \psi - \pi/2)_{3×3} & P_{3×1} \\
0_{1×3} & 1
\end{bmatrix}$$

Where:

$$R(y, \psi - \pi/2) = \begin{bmatrix}
\cos(\psi - \pi/2) & 0 & \sin(\psi - \pi/2) \\
0 & 1 & 0 \\
-\sin(\psi - \pi/2) & 0 & \cos(\psi - \pi/2)
\end{bmatrix}$$

$$P = \begin{bmatrix}
0.04 + 0.113 \cdot \sin(\psi - \pi/2) \\
0 \\
0.034 + 0.113 \cdot \cos(\psi - \pi/2)
\end{bmatrix}$$

### Verification with TF

In the robot, the transformation $^RT_L$ is not calculated by your code: it is published automatically in TF by the following chain:

```mermaid
flowchart LR
    SRV["service<br/>/set_servo_pos"] --> SV[servomoteur]
    SV -- "/set_joint_states<br/>(JointState)" --> JSP[joint_state_publisher]
    JSP --> RSP["robot_state_publisher<br/>(URDF)"]
    RSP -- "TF" --> TF(["base_link → servomotor → lidar"])
```

The `servomotor` node converts the commanded pulse width (in µs) into the joint angle `base_link_to_servomotor`:

$$\theta = 0.623334 + (2000 - \text{pulse}_{\mu s}) \times (-0.001558335) \quad \text{[rad]}$$

**To do:** once the robot has started (see Milestone 3), place the servo at a few positions and compare the published transformation with that obtained by your equations:

```bash
ros2 service call /set_servo_pos jerro_msgs/srv/SetServoPos "{position: 1500}"
ros2 run tf2_ros tf2_echo base_link lidar
```

What connection do you make between the joint angle $\theta$ and the angle $\psi$ of your equations? Do the URDF dimensions match those in Figure 2.6?

## 2.2.3 Milestone 2: What is an action server?

To reconstruct the environment in 3D, the robot must chain together several seconds of work: move the servo, take a scan, start over... We want to be able to **start this scan from another node** (or the command line) and **track its progress**. This is the role of an **action**.

You already know two ways to communicate between nodes. Action is the third:

| | Topic | Service | Action |
|---|---|---|---|
| Exchange | continuous stream, no response | 1 request → 1 response | 1 goal → news… → 1 result |
| Duration | — | short (a few ms) | long (seconds, minutes) |
| In our robot | `/scan` (lidar publishes continuously) | `/set_servo_pos` (move servo) | `/perception/start_scan` (3D scan) |

> An action is a **long** service that **provides updates** while it is working.

This milestone is a reading: you will write the code in Milestone 3.

### The three messages of an action

An action is defined in a `.action` file, in three parts separated by `---`. Here is the one for our scan (`jerro_msgs/action/StartScan.action`):

```
# Goal: what the client requests
bool start
---
# Result: sent once, at the end
sensor_msgs/PointCloud2 point_cloud
bool success
string message
---
# Feedback: sent multiple times, during the work
float32 progress
```

To display it on the robot:
```bash
ros2 interface show jerro_msgs/action/StartScan
```

### Flow of an action

The **client** (here, the command line) sends a goal. The **server** (here, your `lidar_traitement` node) accepts it, sends feedback during the work, then the result:

```mermaid
sequenceDiagram
    participant C as Client<br/>(ros2 action send_goal)
    participant S as Server<br/>(lidar_traitement)
    C->>S: Goal {start: true}
    S-->>C: Goal accepted
    loop for each servo position
        Note over S: move servo,<br/>take a scan
        S-->>C: Feedback {progress: 10 %, 20 %, …}
    end
    S-->>C: Result {success, message, point_cloud}
```

On the server side, a goal goes through the following states:

```mermaid
stateDiagram-v2
    direction LR
    state "Accepted" as Accepted
    state "Executing" as Executing
    state "Succeeded" as Succeeded
    state "Aborted" as Aborted
    [*] --> Accepted : goal received
    Accepted --> Executing : execute_callback starts
    Executing --> Succeeded : goal_handle.succeed()
    Executing --> Aborted : goal_handle.abort()
    Succeeded --> [*]
    Aborted --> [*]
```

ROS 2 also allows the client to **cancel** an action in progress. We will not use it: by default, the server refuses cancellations (see the box below).

### Server side, in code

**1. Create the server**, in the node constructor. We give it the action type, its name, and the function to execute when a goal arrives:

```python
from rclpy.action import ActionServer
from jerro_msgs.action import StartScan

self._action_server = ActionServer(self, StartScan, 'perception/start_scan', self.execute_callback)
```

**2. Write `execute_callback`**. It receives a `goal_handle`, which gives access to the goal (`goal_handle.request`) and allows replying to the client. Generic example, with fictitious work:

```python
def execute_callback(self, goal_handle):
    n_etapes = 10
    for i in range(n_etapes):
        time.sleep(0.5)                          # the work (in Milestone 3: servo + scan)

        feedback = StartScan.Feedback()
        feedback.progress = 100.0 * (i + 1) / n_etapes
        goal_handle.publish_feedback(feedback)   # send updates to the client

    goal_handle.succeed()                        # mark goal as succeeded
    result = StartScan.Result()
    result.success = True
    result.message = "Done"
    return result                                # result is sent to the client
```

Things to remember:
- `publish_feedback` can be called as many times as you want;
- **always** end with `succeed()` (or `abort()` on failure) **then** `return` the result.

### Client side: the command line

```bash
ros2 action list                                    # available actions
ros2 action info /perception/start_scan             # who provides this action?
ros2 action send_goal /perception/start_scan jerro_msgs/action/StartScan "{start: true}" --feedback
```

With `--feedback`, we see the progress scroll, then the result:
```
Goal accepted with ID: 3f2a...
Feedback:
    progress: 10.0
Feedback:
    progress: 20.0
...
Result:
    success: true
    message: Done
Goal finished with status: SUCCEEDED
```

### ⚠️ During `execute_callback`, the node does nothing else

A ROS 2 node processes its callbacks **one at a time**. As long as `execute_callback` is running (several seconds of `time.sleep` and waiting for scans), no other node callback is executed. Three consequences for Milestone 3:

1. **Do not wait for the response of a service** called from `execute_callback`: the response would be processed by the node... which is busy. The program would wait indefinitely (*deadlock*). Therefore, send the request with `call_async` and continue.
2. **The TF listener has its own node and its own thread**: we give it a second node (`self.tf_node`) and `spin_thread=True`, so that transformations keep arriving during the scan. If we gave it `self`, this same node would be busy with `execute_callback` and TF would no longer be updated.
3. **A launched scan cannot be cancelled**: the cancellation request would only be read at the end.

## 2.2.4 Milestone 3: 3D Scan Implementation

You will rewrite the `lidar_traitement.py` node of the `jerro_perception` package so that it offers the `/perception/start_scan` action.

### Overview

```mermaid
flowchart LR
    CLI["ros2 action send_goal"] -- "action<br/>/perception/start_scan" --> LT["lidar_traitement<br/>(your node)"]
    LT -- "service<br/>/set_servo_pos" --> SV[servomoteur]
    SV -- "/set_joint_states" --> JSP[joint_state_publisher]
    JSP --> RSP[robot_state_publisher]
    RSP -- "TF base_link → lidar" --> LT
    LID[sllidar_ros2] -- "topic /scan" --> LT
    LT -- "topic<br/>/perception/lidar_pc2" --> RV[RViz]
```

### The provided tool: the ScanAssembler class

The `laser_assembler` package contains the `ScanAssembler` class. It is a simple Python class (no node to launch): for each added scan, it reads from TF the valid `base_link ← lidar` transformation at the moment of the scan (your $^RT_L$ from Milestone 1), transforms the points into `base_link`, and stores them.

| Method | Role |
|---|---|
| `ScanAssembler(tf_buffer, fixed_frame='base_link')` | Creates the assembler, which will read transformations from `tf_buffer` |
| `add_scan(scan)` | Adds a `LaserScan` (returns `False` if TF is not found) |
| `to_pointcloud2()` | Returns the assembled `PointCloud2` in `base_link` |
| `clear()` | Clears memory before a new scan |

The `frame_id` of the scans must be the name of the lidar frame in the URDF (`lidar`): this is set in the launch file (`frame_id:=lidar` for `sllidar_ros2`).

### Prerequisites and compilation

Clone the `laser_assembler` package (and the lidar driver, if not already done in Lab 2a), then compile:

In [ ]:
%%bash
cd ~/ros2_ws/src
git clone https://github.com/Lab-CORO/laser_assembler_ros2
git clone https://github.com/Slamtec/sllidar_ros2.git   # if not already done in Lab 2a
cd ~/ros2_ws
colcon build --packages-select sllidar_ros2 laser_assembler jerro_perception
source install/setup.bash

### The skeleton to complete

Replace the contents of `jerro_perception/jerro_perception/lidar_traitement.py` with the skeleton below. Parts marked `TODO` are to be written: the following steps guide you, one by one.

⚠️ The `SWEEP_*` constants limit the servo's travel so the lidar doesn't hit the RPI housing. **Do not modify them without the lab instructor's agreement.**

In [ ]:
import time

import rclpy
from rclpy.node import Node
from rclpy.action import ActionServer
from rclpy.wait_for_message import wait_for_message
import tf2_ros

from sensor_msgs.msg import LaserScan, PointCloud2
from jerro_msgs.action import StartScan
from jerro_msgs.srv import SetServoPos

from laser_assembler.scan_assembler import ScanAssembler


class LidarTraitement(Node):
    # Servo positions, in pulse width (µs). DO NOT MODIFY without lab instructor.
    SWEEP_MIN_PULSE = 1300
    SWEEP_MAX_PULSE = 1700
    SWEEP_STEP_PULSE = 20
    CENTER_PULSE = 1500
    SETTLE_TIME = 0.3   # s: time allowed for the servo to reach its position

    def __init__(self):
        super().__init__('lidar_traitement')

        # 3D cloud published at the end of each scan
        self.pc2_pub = self.create_publisher(PointCloud2, 'perception/lidar_pc2', 10)

        # TF: the listener has its own node and its own thread, to stay up to date
        # during execute_callback (see Milestone 2 box)
        self.tf_buffer = tf2_ros.Buffer()
        self.tf_node = rclpy.create_node('lidar_traitement_tf')
        self.tf_listener = tf2_ros.TransformListener(self.tf_buffer, self.tf_node, spin_thread=True)
        self.assembler = ScanAssembler(self.tf_buffer, fixed_frame='base_link')

        # TODO 1: create the client for the 'set_servo_pos' service (type SetServoPos)
        #         and wait for it to be available.

        # TODO 3: create the action server 'perception/start_scan' (type StartScan),
        #         which calls self.execute_callback.

        self.get_logger().info("lidar_traitement ready.")

    def move_servo(self, pulse_us):
        """Sends the servo to the pulse_us position (in µs) and waits for it to get there."""
        # TODO 2
        pass

    def execute_callback(self, goal_handle):
        """Performs a full 3D scan."""
        # TODO 4: sweep from SWEEP_MIN_PULSE to SWEEP_MAX_PULSE and assemble scans.

        # TODO 5: publish the cloud, center the servo again and return the result.
        pass


def main(args=None):
    rclpy.init(args=args)
    node = LidarTraitement()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        pass
    finally:
        node.tf_node.destroy_node()
        node.destroy_node()
        rclpy.shutdown()


if __name__ == '__main__':
    main()

### Step 1: control the servo (TODO 1 and 2)

**In the robot:** the `servomotor` node (`jerro_drivers` package) controls the RPI PWM and provides the `/set_servo_pos` service. Your node only has to send a request to it:

```
int16 position   # pulse width in µs (1000 to 2000, 1500 = center)
---
bool result
```

**To do:**
- **TODO 1**: in the constructor, create a client for this service and wait for it to be available.
- **TODO 2**: in `move_servo`, send the request **without waiting for the response** (see Milestone 2 box), then allow the servo the `SETTLE_TIME` to reach its position.

**Verify:** first test the service manually (`ros2 service call /set_servo_pos jerro_msgs/srv/SetServoPos "{position: 1400}"`). Then temporarily call `self.move_servo(1400)` at the end of the constructor: in RViz, the `lidar` frame should rotate. Then remove this call.

### Step 2: a "dummy" action server (TODO 3)

**In the robot:** before moving hardware, we verify that the action works.

**To do:**
- **TODO 3**: create the action server in the constructor.
- Temporarily write an `execute_callback` that uses the generic example from Milestone 2: a loop with `time.sleep`, one feedback per turn, then `succeed()` and the result.

**Verify:**
```bash
ros2 action send_goal /perception/start_scan jerro_msgs/action/StartScan "{start: true}" --feedback
```
The progress should scroll up to 100%, then `Goal finished with status: SUCCEEDED`.

### Step 3: the real scan (TODO 4)

**In the robot:** we replace the dummy work with the scan. For each servo position, we take **a single** scan, at the moment when the lidar is stationary: this is the role of `wait_for_message` (a subscriber would otherwise receive scans continuously, even while the servo is moving).

```mermaid
flowchart TD
    A(["Goal received"]) --> B["assembler.clear()"]
    B --> C["move_servo : next position"]
    C --> D["wait_for_message : a scan on /scan"]
    D --> E["assembler.add_scan(scan)"]
    E --> F["publish_feedback"]
    F --> G{"Last position?"}
    G -- no --> C
    G -- yes --> H["TODO 5 : end of scan"]
```

**To do (TODO 4):**
- clear the assembler at the start of the scan;
- loop through positions from `SWEEP_MIN_PULSE` to `SWEEP_MAX_PULSE` by steps of `SWEEP_STEP_PULSE` (hint: `range`);
- at each position: move the servo, wait for a scan, add it to the assembler, publish progress (in %).

The first movement (from center to `SWEEP_MIN_PULSE`) takes longer than the others: allow an extra second for the servo.

**Verify:** log the number of assembled scans (`self.assembler.n_scans`) at the end of the loop. It must equal the number of positions.

### Step 4: the result (TODO 5)

**To do:**
- retrieve the point cloud with `to_pointcloud2()` and publish it on `perception/lidar_pc2`;
- return the servo to the center (`CENTER_PULSE`);
- finish the action (`succeed()`) and return the result (`success` and `message`).

Do **not** put the cloud in the `point_cloud` field of the result: `ros2 action send_goal` would print its ~66,000 points in the terminal, which is very slow on the RPI. The point cloud is viewed in RViz, on `/perception/lidar_pc2`. The `point_cloud` field will be used in laboratory 2c.

### Help Sheet

| Need | API |
|---|---|
| Service client | `self.create_client(Type, 'name')`, `client.wait_for_service()`, `client.call_async(request)` |
| Create a request | `req = SetServoPos.Request()` then `req.position = ...` |
| Action server | `ActionServer(self, Type, 'name', self.execute_callback)` |
| Inside `execute_callback` | `goal_handle.publish_feedback(fb)`, `goal_handle.succeed()`, `return result` |
| Action messages | `StartScan.Feedback()`, `StartScan.Result()` |
| Receive a single message | `ok, msg = wait_for_message(LaserScan, self, '/scan')` |
| Assembler | `self.assembler.clear()`, `.add_scan(scan)`, `.to_pointcloud2()`, `.n_scans` |
| Publish | `self.pc2_pub.publish(msg)` |

### Final test with RViz

⚠️ **Before testing your code, show it to the lab instructor for authorization!** Errors in servomotor commands can cause hardware damage.

1. Connect to the RPI via VNC
2. Start the pigpio daemon (required for the `servomotor` node):

In [ ]:
%%bash
sudo pigpiod

3. Run the launch file. It starts the robot description (TF), the servomotor, the lidar, the camera, and your two nodes (`cam_traitement` and `lidar_traitement`):

> In the launch file, do **not** give a `name=` to the `lidar_traitement` node: this parameter would rename all nodes in the process, including `lidar_traitement_tf`.

In [ ]:
%%bash
ros2 launch jerro_perception lidar_et_cam.launch.py

4. Run rviz:

In [ ]:
%%bash
ros2 run rviz2 rviz2

5. **Configuration in RViz:**
   - "Fixed Frame": `base_link`
   - Add "TF", "LaserScan", and "PointCloud2"
   - For TF: select `base_link`, `servomotor`, and `lidar`
   - For LaserScan: topic `/scan`
   - For PointCloud2: topic `/perception/lidar_pc2`

6. **Start acquisition:**

In [ ]:
%%bash
ros2 action send_goal /perception/start_scan jerro_msgs/action/StartScan "{start: true}" --feedback

### Expected result

You should observe:
- The rotation of the `lidar` frame relative to `base_link`
- The 2D lidar data as it goes
- The complete 3D point cloud once the scan is finished

![Figure 2.9 - RViz Result](figure_2_9.png)
*Figure 2.9: Preview of the expected result type, in the "rviz" application*

## 2.2.5 Demonstration and Questions

### Demonstration to the lab instructor

The demonstration and additional questions will be completed at the end of the **next session (laboratory #2c)**.

### Points to check

- [ ] Homogeneous transformation $^RT_L$ developed and compared with the `base_link → lidar` TF
- [ ] Servo control via the `/set_servo_pos` service
- [ ] Functional `/perception/start_scan` action server, with feedback
- [ ] 3D point cloud visible in RViz

### To go further

- With the lab instructor's agreement, modify `SWEEP_STEP_PULSE` and `SETTLE_TIME`
- Observe the impact on the density and quality of the point cloud, as well as on the scan duration